# Which ad channels pay back more than a dollar

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Bayesian Linear Regression** · a worked example from the StatNex teaching guide

**The story.** A small online brand recorded 30 weeks of gross profit (before advertising costs) and its spending on three channels: social media ads, search ads and leaflets. The budget for each channel changed from week to week. The owner wants to know which channels pay for themselves.

**The question.** Which channels bring back more than a dollar of gross profit for each dollar they cost?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `pymc` and `arviz`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "pymc>=6.3" "arviz>=1.3"

## The table

This cell saves the table `14_bayesian/bayes_regression.csv` (30 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Week,Gross profit ($000),Social media ads ($000),Search ads ($000),Leaflets ($000)
1.0,35.5,5.5,2.6,1.6
2.0,29.7,2.1,1.2,0.6
3.0,38.9,5.1,4.7,3.4
4.0,37.2,2.0,5.6,1.2
5.0,38.2,3.1,4.5,3.6
6.0,34.7,1.7,4.4,2.4
7.0,34.5,3.3,3.2,1.4
8.0,36.2,1.5,4.2,3.7
9.0,26.4,2.9,1.4,1.3
10.0,38.8,5.8,3.4,3.9
11.0,45.9,5.5,5.9,1.7
12.0,32.2,4.0,1.3,2.4
13.0,28.8,3.3,1.3,0.6
14.0,41.6,5.3,5.5,1.8
15.0,37.7,5.5,2.7,2.3
16.0,34.1,5.7,1.1,2.7
17.0,40.2,3.2,4.1,3.4
18.0,39.6,2.2,5.9,2.1
19.0,35.2,4.2,2.5,2.1
20.0,44.0,5.3,3.6,3.8
21.0,45.0,1.9,5.9,2.9
22.0,35.4,2.9,3.6,2.8
23.0,31.6,3.4,1.7,1.5
24.0,36.7,2.5,2.8,2.7
25.0,32.1,4.8,2.7,3.4
26.0,43.6,6.0,4.0,2.3
27.0,39.8,3.3,4.9,1.3
28.0,37.3,5.7,2.2,0.8
29.0,38.1,2.9,5.9,0.7
30.0,37.4,1.1,5.1,3.3

## Libraries

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import statsmodels.api as sm

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
# A row missing any of the chosen columns is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Gross profit ($000)"
predictors = ["Social media ads ($000)", "Search ads ($000)", "Leaflets ($000)"]
credible = 95 / 100
comparison = 1
low, high = (1 - credible) / 2, (1 + credible) / 2
frame = data[[response, *predictors]].apply(pd.to_numeric, errors="coerce").dropna()
y = frame[response].to_numpy(float)
x = sm.add_constant(frame[predictors].to_numpy(float), has_constant="add")
n, p = x.shape

## Posterior of the coefficients: PyMC

In [ ]:
# The model: y = b0 + b1 x1 + ... + bk xk + error, the errors independent and normal with
# SD sigma. The prior is the non-informative p(b, sigma^2) ∝ 1/sigma^2: flat for every
# coefficient and for log(sigma). Under it each coefficient's posterior is a t centred on
# the least-squares estimate, so its credible interval is the confidence interval of
# multiple regression; PyMC samples it by MCMC (NUTS), starting from those estimates.
start = sm.OLS(y, x).fit()
with pm.Model():
    beta = pm.Flat("beta", shape=p)
    log_sigma = pm.Flat("log_sigma")  # flat in log(sigma): p(sigma^2) ∝ 1/sigma^2
    sigma = pm.Deterministic("sigma", pm.math.exp(log_sigma))
    pm.Normal("y", mu=pm.math.dot(x, beta), sigma=sigma, observed=y)
    # The coefficients are correlated, so the sampler learns a full mass matrix
    # (init="jitter+adapt_full") rather than one scale per coefficient; it mixes far faster.
    trace = pm.sample(draws=25_000, tune=2_000, chains=4, random_seed=2026, progressbar=False,
                      init="jitter+adapt_full", initvals={"beta": start.params, "log_sigma": np.log(np.sqrt(start.scale))})
print(az.summary(trace, var_names=["beta", "sigma"], kind="stats", ci_prob=credible, ci_kind="eti"))

betas = trace.posterior["beta"].to_numpy().reshape(-1, p)
sigmas = trace.posterior["sigma"].to_numpy().ravel()
# R squared describes the fit of the least-squares line; it is not a posterior quantity.
fit = pd.Series({"cases": n, "explanatory variables": p - 1, "degrees of freedom": n - p,
                 "R squared": start.rsquared, "sigma mean": sigmas.mean(), "sigma median": np.median(sigmas),
                 "sigma lower": np.quantile(sigmas, low), "sigma upper": np.quantile(sigmas, high),
                 "credible level": credible})
# P(b > c) is the share of draws above the comparison value. For the constant it asks
# nothing useful, so it is left out.
coefficients = pd.DataFrame({
    "mean": betas.mean(axis=0), "sd": betas.std(axis=0, ddof=1),
    "lower": np.quantile(betas, low, axis=0), "upper": np.quantile(betas, high, axis=0),
    f"P(b > {comparison:g})": [np.nan, *np.mean(betas[:, 1:] > comparison, axis=0)],
}, index=["Constant", *predictors])

print("\nModel summary")
print(fit.to_string())
print(f"\nPosterior of the regression coefficients ({credible:.0%} credible intervals)")
print(coefficients)

## What the result says

Each regression coefficient estimates the gross profit that one more dollar on a channel brings back. A channel pays for itself when its coefficient is above 1. The posterior mean is the best estimate after seeing the 30 weeks. The 95% credible interval is the range that holds the true value with 95% probability.

Search ads bring back 2.41 (interval 1.84 to 2.98), and the probability of more than 1 is 0.99999. Social media ads bring back 1.36 (0.77 to 1.94), with a probability of 0.89. Leaflets bring back 0.74 (-0.12 to 1.61), with a probability of only 0.27.

R² is 0.79: the three channels together explain about 79% of the weekly changes in profit. On this model, search ads almost certainly pay for themselves, social media ads probably do, and leaflets probably do not.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.